# DSCI 552 - Homework 3

**Name:** Zexin Xiong  
**GitHub Username:** zexinxiong  
**USC ID:** 6510910225



In [22]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

RANDOM_SEED = 552

# Support execution from either the repository root or the notebook directory.
candidate_roots = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(root for root in candidate_roots if (root / "data" / "AReM").exists())
DATA_DIR = REPO_ROOT / "data" / "AReM"

SIGNAL_COLUMNS = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23",
]
ALL_COLUMNS = ["time", *SIGNAL_COLUMNS]

print(f"Repository root: {REPO_ROOT.name}/")
print(f"Data directory:  {DATA_DIR.relative_to(REPO_ROOT)}")

Repository root: DSCI552-HW3/
Data directory:  data/AReM


### 1(a) Load and inspect the AReM data

The source files use either commas or whitespace as delimiters. Two files also
contain a trailing comma on the final row. The loader reads only the seven
documented columns, converts every field to numeric values, and raises an error
if required measurements are missing. It does not overwrite any source file.

In [23]:
def dataset_number(path: Path) -> int:
    # Return the integer in a filename such as dataset14.csv.
    match = re.search(r"dataset(\d+)\.csv$", path.name)
    if match is None:
        raise ValueError(f"Unexpected dataset filename: {path.name}")
    return int(match.group(1))


def read_recording(path: Path) -> pd.DataFrame:
    # Read one AReM recording without changing the source file.
    data_lines = [
        line for line in path.read_text(errors="replace").splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]
    if not data_lines:
        raise ValueError(f"No observations found in {path}")

    separator = "," if "," in data_lines[0] else r"\s+"
    frame = pd.read_csv(
        path,
        comment="#",
        header=None,
        names=ALL_COLUMNS,
        usecols=range(len(ALL_COLUMNS)),
        sep=separator,
        engine="python",
    )

    frame = frame.apply(pd.to_numeric, errors="coerce")
    if frame[ALL_COLUMNS].isna().any().any():
        missing = int(frame[ALL_COLUMNS].isna().sum().sum())
        raise ValueError(f"{path} contains {missing} missing/non-numeric values")
    return frame


files = sorted(
    DATA_DIR.glob("*/*.csv"),
    key=lambda path: (path.parent.name, dataset_number(path)),
)

inventory_rows = []
recordings = {}
for path in files:
    frame = read_recording(path)
    activity = path.parent.name
    number = dataset_number(path)
    recordings[(activity, number)] = frame
    inventory_rows.append(
        {
            "activity": activity,
            "dataset": number,
            "rows": len(frame),
            "signals": len(SIGNAL_COLUMNS),
            "source_file": path.relative_to(REPO_ROOT).as_posix(),
        }
    )

inventory = pd.DataFrame(inventory_rows)
inventory_summary = (
    inventory.groupby("activity", sort=True)
    .agg(instances=("dataset", "count"), min_rows=("rows", "min"), max_rows=("rows", "max"))
)

display(inventory_summary)
print(f"Total instances: {len(inventory)}")
print(f"Row counts observed: {sorted(inventory['rows'].unique())}")
assert len(inventory) == 88
assert inventory["signals"].eq(6).all()
assert inventory["rows"].isin([479, 480]).all()

,instances,min_rows,max_rows
activity,,,
bending1,7,480,480
bending2,6,480,480
cycling,15,480,480
lying,15,480,480
sitting,15,479,480
standing,15,480,480
walking,15,480,480


Total instances: 88
Row counts observed: [np.int64(479), np.int64(480)]


The 88 expected instances are present. All files contain six sensor signals.
Most recordings have 480 observations; `sitting/dataset8.csv` has 479 source
rows. The assignment notes that minor data cleaning may be necessary. I retain
all available numeric observations rather than inventing a missing row.

### 1(b) Construct the train/test split

- `bending1` and `bending2`: datasets 1 and 2 are test data.
- All other activities: datasets 1, 2, and 3 are test data.
- Every remaining instance is training data.

In [24]:
def split_name(activity: str, number: int) -> str:
    test_numbers = {1, 2} if activity in {"bending1", "bending2"} else {1, 2, 3}
    return "test" if number in test_numbers else "train"


inventory["split"] = [
    split_name(activity, number)
    for activity, number in zip(inventory["activity"], inventory["dataset"])
]

split_summary = pd.crosstab(inventory["activity"], inventory["split"])
split_summary.loc["Total"] = split_summary.sum(axis=0)
display(split_summary)

assert int((inventory["split"] == "test").sum()) == 19
assert int((inventory["split"] == "train").sum()) == 69

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12
Total,19,69


### 1(c)(i) Common time-domain features

Common time-domain features include:

- **Location:** mean, median, trimmed mean.
- **Extrema and range:** minimum, maximum, peak-to-peak range.
- **Quantiles:** first and third quartiles and interquartile range.
- **Dispersion:** variance, standard deviation, median absolute deviation, and
  root mean square.
- **Distribution shape:** skewness and kurtosis.
- **Dynamics:** first differences, mean/maximum absolute change, slope, and
  zero-crossing or threshold-crossing counts.
- **Temporal dependence:** lagged autocorrelation and related correlation
  summaries.

These feature families are consistent with large-scale time-series feature
libraries such as **tsfresh** and **hctsa**.

References:

- Christ, M., Braun, N., Neuffer, J., & Kempa-Liehr, A. W. (2018).
  *Time Series FeatuRe Extraction on basis of Scalable Hypothesis tests
  (tsfresh - A Python package).* Neurocomputing, 307, 72-77.
  https://doi.org/10.1016/j.neucom.2018.03.067
- Fulcher, B. D., & Jones, N. S. (2017). *hctsa: A Computational Framework for
  Automated Time-Series Phenotyping Using Massive Feature Extraction.* Cell
  Systems, 5(5), 527-531.e3. https://doi.org/10.1016/j.cels.2017.10.001

### 1(c)(ii) Extract the required features

For each of the six signals in each instance, I extract minimum, maximum,
mean, median, sample standard deviation, first quartile, and third quartile.
This produces 42 features per instance. The analysis uses the signals on their
original scales, as allowed by the assignment.

In [25]:
STATISTICS = ["min", "max", "mean", "median", "std", "1st_quart", "3rd_quart"]


def extract_features(frame: pd.DataFrame) -> dict[str, float]:
    result = {}
    for signal_index, column in enumerate(SIGNAL_COLUMNS, start=1):
        values = frame[column]
        result[f"min{signal_index}"] = values.min()
        result[f"max{signal_index}"] = values.max()
        result[f"mean{signal_index}"] = values.mean()
        result[f"median{signal_index}"] = values.median()
        result[f"std{signal_index}"] = values.std(ddof=1)
        result[f"1st_quart{signal_index}"] = values.quantile(0.25)
        result[f"3rd_quart{signal_index}"] = values.quantile(0.75)
    return result


feature_rows = []
for row in inventory.itertuples(index=False):
    features = extract_features(recordings[(row.activity, row.dataset)])
    feature_rows.append(
        {
            "activity": row.activity,
            "dataset": row.dataset,
            "split": row.split,
            **features,
        }
    )

feature_table = pd.DataFrame(feature_rows)
feature_columns = [
    f"{statistic}{signal_index}"
    for signal_index in range(1, 7)
    for statistic in STATISTICS
]

display(feature_table.head().round(3))
print(f"Feature table shape (including 3 metadata columns): {feature_table.shape}")
print(f"Number of extracted numeric features: {len(feature_columns)}")

assert feature_table.shape == (88, 45)
assert len(feature_columns) == 42
assert feature_table[feature_columns].notna().all().all()

,activity,dataset,split,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1,1,test,37.25,45.00,40.625,40.50,1.477,39.25,42.00,0.0,1.30,0.359,0.43,0.323,0.00,0.50,4.0,29.50,19.041,19.250,4.463,16.00,23.250,0.0,7.23,0.833,0.50,0.966,0.00,1.120,27.25,38.25,34.311,35.0,2.188,33.000,36.00,0.0,1.92,0.571,0.43,0.583,0.0,1.30
1,bending1,2,test,38.00,45.67,42.813,42.50,1.436,42.00,43.67,0.0,1.22,0.372,0.47,0.289,0.00,0.50,2.0,29.50,20.097,21.000,3.894,19.00,22.250,0.0,5.76,0.876,0.50,1.047,0.00,1.145,27.67,38.50,33.025,33.0,1.995,32.000,34.50,0.0,3.11,0.571,0.43,0.601,0.0,1.30
2,bending1,3,train,35.00,47.40,43.954,44.33,1.559,43.00,45.00,0.0,1.70,0.426,0.47,0.339,0.00,0.50,6.5,29.75,22.122,23.000,3.031,19.75,24.000,0.0,4.44,0.497,0.43,0.551,0.00,0.830,29.00,38.50,35.588,36.0,2.000,35.362,36.50,0.0,1.79,0.493,0.43,0.514,0.0,0.94
3,bending1,4,train,33.00,47.75,42.180,43.50,3.671,39.15,45.00,0.0,3.00,0.696,0.50,0.631,0.00,1.12,8.5,30.00,22.184,23.000,3.810,20.50,24.372,0.0,5.15,0.990,0.83,0.954,0.43,1.300,20.00,38.67,33.494,35.0,3.849,30.458,36.33,0.0,2.18,0.614,0.50,0.524,0.0,1.00
4,bending1,5,train,33.00,45.75,41.678,41.75,2.243,41.33,42.75,0.0,2.83,0.536,0.50,0.405,0.43,0.71,3.0,28.25,19.007,19.125,4.087,16.50,22.062,0.0,6.42,0.842,0.50,0.929,0.43,1.120,23.67,37.50,29.857,30.0,2.411,28.458,31.25,0.0,1.79,0.383,0.43,0.389,0.0,0.50


Feature table shape (including 3 metadata columns): (88, 45)
Number of extracted numeric features: 42


### 1(c)(iii) Standard deviations and 90% bootstrap confidence intervals

For every extracted feature, the point estimate is the sample standard
deviation across the 88 instances. I construct a nonparametric percentile
bootstrap interval by resampling the 88 instances with replacement 2,000
times. The 5th and 95th percentiles of the bootstrap standard deviations form
the 90% interval. A fixed random seed makes the result reproducible.

In [26]:
def bootstrap_std_interval(
    values: np.ndarray,
    confidence_level: float = 0.90,
    n_resamples: int = 2_000,
    seed: int = RANDOM_SEED,
) -> tuple[float, float, float]:
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    bootstrap_stds = values[sample_indices].std(axis=1, ddof=1)
    alpha = 1.0 - confidence_level
    lower, upper = np.quantile(bootstrap_stds, [alpha / 2, 1 - alpha / 2])
    return values.std(ddof=1), lower, upper


bootstrap_rows = []
for feature_index, feature in enumerate(feature_columns):
    estimate, lower, upper = bootstrap_std_interval(
        feature_table[feature].to_numpy(),
        seed=RANDOM_SEED + feature_index,
    )
    match = re.match(r"(.+?)([1-6])$", feature)
    bootstrap_rows.append(
        {
            "feature": feature,
            "statistic": match.group(1),
            "signal": int(match.group(2)),
            "std_estimate": estimate,
            "ci_90_lower": lower,
            "ci_90_upper": upper,
        }
    )

bootstrap_summary = pd.DataFrame(bootstrap_rows).set_index("feature")
display(bootstrap_summary[["std_estimate", "ci_90_lower", "ci_90_upper"]].round(3))

assert len(bootstrap_summary) == 42
assert (bootstrap_summary["ci_90_lower"] <= bootstrap_summary["std_estimate"]).all()
assert (bootstrap_summary["std_estimate"] <= bootstrap_summary["ci_90_upper"]).all()

,std_estimate,ci_90_lower,ci_90_upper
feature,,,
min1,9.570,8.253,10.761
max1,4.394,3.422,5.326
mean1,5.336,4.711,5.865
median1,5.440,4.786,5.983
std1,1.772,1.573,1.955
1st_quart1,6.154,5.572,6.648
3rd_quart1,5.139,4.345,5.845
min2,0.000,0.000,0.000
max2,5.063,4.638,5.407


The intervals quantify sampling uncertainty in each across-instance standard
deviation. Wide intervals indicate that the estimated spread is sensitive to
which activity instances appear in the sample. Because the features use
different physical scales, their raw standard deviations should not be used as
a direct ranking of predictive importance.

### 1(c)(iv) Three important time-domain feature families

I would retain **mean**, **standard deviation**, and **maximum** as the three
most useful feature families for the next classification stage.

1. **Mean** summarizes the typical sensor level over the full activity window.
   It is less sensitive than an extreme value to one unusual observation.
2. **Standard deviation** measures within-window movement variability. Two
   activities can have similar average positions but very different motion
   intensity.
3. **Maximum** captures peak posture or movement values that the mean and
   standard deviation may smooth away.

Together they represent level, variability, and extremes. This is a judgment
based on complementary information rather than a claim that raw feature scale
equals predictive importance. The next assignment can test the choice using
cross-validation and regularized classification.

In [27]:
selected_feature_families = ["mean", "std", "max"]
selected_columns = [
    feature
    for feature in feature_columns
    if any(feature.startswith(family) for family in selected_feature_families)
]

selected_summary = feature_table.groupby("activity")[selected_columns].mean()
display(selected_summary.round(2))
print(f"Selected feature columns for six signals: {len(selected_columns)}")
assert len(selected_columns) == 18

,max1,mean1,std1,max2,mean2,std2,max3,mean3,std3,max4,mean4,std4,max5,mean5,std5,max6,mean6,std6
activity,,,,,,,,,,,,,,,,,,
bending1,46.80,42.67,1.91,1.88,0.45,0.37,28.62,19.16,3.84,6.31,0.79,0.89,36.52,30.20,2.61,2.72,0.55,0.53
bending2,44.96,29.42,4.54,6.39,0.45,0.66,27.36,15.33,4.98,6.85,0.76,0.90,32.72,18.73,4.87,7.23,0.98,0.91
cycling,44.48,36.75,3.77,10.85,2.69,1.89,26.39,16.40,3.47,9.22,2.99,1.70,24.49,17.40,3.03,9.18,3.04,1.75
lying,42.73,39.85,2.02,2.44,0.34,0.29,14.72,7.26,3.04,4.20,0.58,0.61,17.59,9.26,3.60,4.04,0.66,0.56
sitting,46.36,41.75,2.05,3.28,0.40,0.37,23.53,14.86,4.62,5.93,0.75,0.84,25.20,16.10,4.44,6.09,0.90,0.82
standing,47.12,43.86,2.19,2.90,0.40,0.37,21.32,14.77,3.16,5.58,0.62,0.71,21.08,13.70,3.43,5.31,0.91,0.71
walking,46.26,34.43,4.79,14.00,4.34,2.44,23.57,15.39,2.91,9.05,3.20,1.62,24.18,16.03,3.10,9.13,3.34,1.70


Selected feature columns for six signals: 18


## 2. ISLR Exercise 3.7.4

Suppose a data set contains one predictor and a quantitative response. We fit
both a linear regression and a cubic regression,
$Y = \beta_0 + \beta_1X + \beta_2X^2 + \beta_3X^3 + \epsilon$.

### (a) Training RSS when the true relationship is linear

The cubic regression will have training RSS no larger than the linear
regression. The linear model is nested inside the cubic model: the cubic model
can reproduce it by setting $\beta_2 = \beta_3 = 0$. Adding predictors cannot
increase the least-squares training RSS. It will usually decrease it slightly,
even when the extra terms only fit noise.

### (b) Test RSS when the true relationship is linear

The linear regression is expected to have lower test RSS. It has the correct
functional form and estimates fewer parameters, so it has lower variance. The
cubic model adds unnecessary flexibility and is more likely to fit random
training noise. For one particular random test set either model could happen to
win, but the linear model should perform better on average.

### (c) Training RSS when the true relationship is not linear

The cubic regression still has training RSS no larger than the linear
regression because the models remain nested. This statement does not depend on
the true relationship.

### (d) Test RSS when the true relationship is not linear

There is not enough information to decide. The cubic regression may have lower
test RSS if its additional curvature approximates the true relationship well.
If the nonlinearity is weak or has a form that a cubic polynomial does not
capture, the cubic model's higher variance may outweigh its reduction in bias.
The answer depends on the unknown relationship and the amount of data.

## Reproducibility checks

The checks below summarize the required data and outputs. They intentionally
fail if files are missing, the split is wrong, or the feature table is
incomplete.

In [28]:
checks = pd.Series(
    {
        "Seven activity folders": inventory["activity"].nunique() == 7,
        "Eighty-eight instances": len(inventory) == 88,
        "Six signals per instance": inventory["signals"].eq(6).all(),
        "Nineteen test instances": (inventory["split"] == "test").sum() == 19,
        "Sixty-nine training instances": (inventory["split"] == "train").sum() == 69,
        "Forty-two extracted features": len(feature_columns) == 42,
        "No missing extracted features": feature_table[feature_columns].notna().all().all(),
        "Forty-two bootstrap intervals": len(bootstrap_summary) == 42,
    },
    name="passed",
)

display(checks.to_frame())
assert checks.all(), "At least one reproducibility check failed."
print("All reproducibility checks passed.")

,passed
Seven activity folders,True
Eighty-eight instances,True
Six signals per instance,True
Nineteen test instances,True
Sixty-nine training instances,True
Forty-two extracted features,True
No missing extracted features,True
Forty-two bootstrap intervals,True


All reproducibility checks passed.
